<a href="https://colab.research.google.com/github/PauloRadatz/opendss-python-examples/blob/main/presentations/ieee_et_pes_pels_joint_chapter_workshop/01_opendss_workflow_with_py_dss_interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 — OpenDSS workflow with py-dss-interface

The goal of this notebook is simple: take the workflow we usually run from a Run.dss file using OpenDSS standalone version and reproduce it from Python.

The key idea is that **OpenDSS is controlled through text commands**. The `compile`, `New`, `Edit`, `set`, `solve`, `show`, and `export` commands you type in the OpenDSS standalone are exactly the same strings you can send from Python through `dss.text(...)`. That is what `py-dss-interface` does — it gives Python a direct line to OpenDSS so we can drive the simulation from a script instead of typing in the GUI.

In this notebook we use **only** `py-dss-interface`. That is intentional: I want the bridge between the run file and Python to feel obvious. In Notebook 4 we will see how `py-dss-toolkit` makes a lot of this work shorter.

## Setup

Install `py-dss-interface` and clone the repository so we have access to the `IEEE123bus` test feeder.

In [ ]:
!pip install py-dss-interface
!git clone https://github.com/PauloRadatz/opendss-python-examples.git
%cd opendss-python-examples

Cloning into 'opendss-python-examples'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 154 (delta 52), reused 99 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 1019.69 KiB | 4.45 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/opendss-python-examples/feeder_models/IEEETestCases/123Bus/opendss-python-examples


## Two files, two responsibilities

When we work with OpenDSS we usually deal with two files:

- **Master file** — defines the feeder model: circuit, transformers, lines, loads, regulators, generators. There is no simulation here, just the model.
- **Run file** — loads the master file, sets simulation options, solves, and asks for results.

Think of it this way: the master file is *what the feeder is*, and the run file is *what we want to do with it*.

What we will do in this notebook is take the run file and replace it with Python code that sends the same commands.

In [ ]:
import pathlib
import py_dss_interface

dss_file= "/content/opendss-python-examples/feeder_models/IEEETestCases/123Bus/IEEE123Master.dss"

print(f"Master file: {dss_file}")

Master file: /content/opendss-python-examples/feeder_models/IEEETestCases/123Bus/IEEE123Master.dss


## Create the OpenDSS object

`py_dss_interface.DSS()` starts an OpenDSS engine inside Python. From now on, every command we send through `dss.text(...)` is the same string we would type in the OpenDSS standalone.

In [ ]:
dss = py_dss_interface.DSS()
print(f"OpenDSS started: {dss.started}")
print(f"OpenDSS version: {dss.dssinterface.version}")

OpenDSS started: True
OpenDSS version: Version 10.1.0.1 @ C++ (64-bit build); license status: Open; built on Feb 16 2026 12:36:32


## Compile the master file

`compile` loads the master file and runs every command it contains. After this step the feeder model lives in OpenDSS memory, but we have not solved anything yet.

**Important:** We must pass the **full path** to `compile`. We can confirm that OpenDSS now points to the correct folder when using `dss.dssinterface.datapath`. Let's set show reports option to False since in Google Colab we can't open a external text editor as we can do when using Windows.

In [ ]:
dss.text(f"compile [{dss_file}]")
dss.text("set showreports=false")

# Confirm the OpenDSS data path now points to 123bus.
print(f"OpenDSS data path: {dss.dssinterface.datapath}")

OpenDSS data path: /content/opendss-python-examples/feeder_models/IEEETestCases/123Bus/


## Send the same commands as the run file

Below is what `Run.dss` (file located in the repo folder) does for a snapshot solve, translated one line at a time. Each `dss.text(...)` call is exactly the string we would type in OpenDSS.

In [ ]:
dss.text("new Energymeter.m element=line.l115 terminal=1")
dss.text("solve")

dss.text("Show Voltage LN Nodes")


''

## The text method also returns strings

Some OpenDSS commands return useful information when we send them through `dss.text(...)`. `export voltages`, for example, writes a CSV file and returns the path to it. That can be handy when we want to capture an OpenDSS report without writing the file logic ourselves.

In [ ]:
export_path = dss.text("export voltages")
print(f"Voltages exported to: {export_path}")

Voltages exported to: /content/opendss-python-examples/feeder_models/IEEETestCases/123Bus/ieee123_EXP_VOLTAGES.csv


## OpenDSS Verbs Not Supported in Google Colab

Because Google Colab uses the Linux version of OpenDSS, the `plot` and `visualize` verbs are not available. In the Windows version of OpenDSS, these verbs rely on `DSSView.exe` for plotting, which is exclusively available on Windows. However, you will be able to generate plots using Python when utilizing `py-dss-toolkit`.

## Why Use Python?

You might be wondering: why use Python if it's just sending the same commands as OpenDSS standalone? One major benefit is that we can now capture information directly in memory. For instance, simulation results can be stored as Python variables and used as desired, without needing to export and re-import files. More advanced benefits, like automation and custom analysis, will be demonstrated in the upcoming notebooks.

For instance, now that the power flow is solved, we read results directly from `py-dss-interface`. For example:

- `dss.circuit.total_power` — total active and reactive power at the feederhead in kW and kvar.
- `dss.circuit.buses_vmag_pu` and `dss.circuit.nodes_names` — bus voltages in per unit and the matching node names.

In [ ]:
p_kw, q_kvar = dss.circuit.total_power
p_loss_w, q_loss_var = dss.circuit.losses

print(f"Feederhead P: {-p_kw:>12,.1f} kW")
print(f"Feederhead Q: {-q_kvar:>12,.1f} kvar")

Feederhead P:      3,615.2 kW
Feederhead Q:      1,311.5 kvar


In [ ]:
voltages_pu = dss.circuit.buses_vmag_pu
node_names = dss.circuit.nodes_names

print(f"Number of nodes: {len(voltages_pu)}")
print(f"Min voltage: {min(voltages_pu):.4f} pu")
print(f"Max voltage: {max(voltages_pu):.4f} pu")

Number of nodes: 278
Min voltage: 0.9792 pu
Max voltage: 1.0500 pu


## Key takeaways

- The **master file** is the feeder model. The **run file** is the simulation script. Keep them separated.
- Anything you can do in OpenDSS, you can do from Python via `dss.text(...)`. That is the bridge.
- `py-dss-interface` allows you to do all you do with OpenDSS standalone but also provides several functions that allow you to access information in memory without the need of exporting files.
- Use the **full path** in `compile` so your script does not depend on the OpenDSS data path.

OpenDSS works fine on its own. What Python adds — and what we will see in the next notebooks — is the ability to *automate* the run file. Loops, conditionals, comparisons, and visualizations all become natural once the simulation lives inside a Python script.

## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)

## Contact

For questions or follow-up about these materials:

- Paulo Radatz
- Email: [paulo.radatz@gmail.com](mailto:paulo.radatz@gmail.com)
- LinkedIn: [linkedin.com/in/pauloradatz](https://www.linkedin.com/in/pauloradatz/)
- Website: [pauloradatz.me](https://www.pauloradatz.me/)